# PageIndex: Multi-Modal Extraction of Ionic-Conductivity Data from Scientific Papers

**A pipeline that extracts structured measurements from text, tables, *and* figures — with an agentic verifier that recovers failed extractions.**

This notebook demonstrates the full PageIndex extraction pipeline:

1. **Quick Start** — Extract measurements from a paper in 3 lines of code
2. **Inspect the Schema** — See exactly what fields are extracted and why
3. **Inspect the Prompts** — Read the LLM prompts that drive extraction
4. **Run Extraction** — Process a real paper end-to-end
5. **Agentic Verification** — Watch the verifier agent recover missed data
6. **Explore Results** — Tables, filtering, and export

## 0. Setup

Set your Gemini API key and ensure dependencies are available. If running on Colab, uncomment the install line.

In [ ]:
# Uncomment for Colab:
# !pip install google-genai pydantic python-dotenv Pillow spacy pandas matplotlib
# !python -m spacy download en_core_web_sm

import os
os.environ["GEMINI_API_KEY"] = "YOUR_KEY_HERE"  # <-- replace with your key

# If running locally with a .env file, this is handled automatically.

---
## 1. Quick Start — Extract in 3 Lines

The simplest way to use PageIndex. Point it at a paper's markdown file (produced by document ingestion) and get structured measurements back.

In [ ]:
from parse import extract

# Point to any paper markdown file (with extracted images in the same folder)
paper = "../output/downselectedpapers_jiyoung/Composite Polymer Electrolytes with Li7La3Zr2O12 Garnet-Type Nanowires as Ceramic Fillers/Composite Polymer Electrolytes with Li7La3Zr2O12 Garnet-Type Nanowires as Ceramic Fillers.md"

result = extract(paper)
print(result.summary)

In [ ]:
# View as a formatted table
print(result.markdown)

In [ ]:
# Or as a pandas DataFrame for analysis
df = result.dataframe
df[["canonical_formula", "material_class", "normalized_conductivity", 
    "normalized_temperature_c", "source", "confidence"]].head(10)

---
## 2. Inspect the Extraction Schema

PageIndex extracts measurements into a structured `MeasuredPoint` schema. Each field has a description that guides the LLM.  You can inspect the full schema programmatically.

In [ ]:
from parse import show_schema

# Print a human-readable overview of every field
show_schema()

In [ ]:
# Get the raw JSON Schema (useful for documentation or validation)
import json
schema_json = show_schema(as_json=True)
print(json.loads(schema_json).keys())  # top-level keys
# Full schema is available as: json.loads(schema_json)

In [ ]:
# Access the Pydantic model class directly
from parse import get_schema_model

MeasuredPoint = get_schema_model()

# Inspect field names, types, and defaults
for name, field_info in MeasuredPoint.model_fields.items():
    req = "required" if field_info.is_required() else "optional"
    print(f"  {name:<35} ({req})")

---
## 3. Customizing the Schema

The default schema targets ionic conductivity in polymer-ceramic composites. But you can **extend or modify** the schema to extract additional fields — or adapt it for an entirely different domain.

Below we show two examples:
- **Extending**: Adding fields to the existing schema (e.g., electrode material, cycle life)
- **Replacing**: Defining a new schema for a different extraction target (e.g., mechanical properties)

### 3a. Extending the existing schema

Add new fields to `MeasuredPoint` without touching the original code. Pydantic model inheritance makes this straightforward — the new fields are included in the JSON schema sent to the LLM, so the model will attempt to populate them.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List
from parse import get_schema_model

# Get the base schema
MeasuredPoint = get_schema_model()

# Extend it with new fields
class ExtendedMeasuredPoint(MeasuredPoint):
    """Extended schema: adds battery cycling and electrode fields."""
    
    electrode_material: Optional[str] = Field(
        None, 
        description="Electrode material used in the cell test (e.g., 'LiFePO4', 'Li metal', 'NCM811')."
    )
    cycle_count: Optional[int] = Field(
        None,
        description="Number of charge-discharge cycles reported for this measurement."
    )
    capacity_retention_pct: Optional[float] = Field(
        None,
        description="Capacity retention as a percentage after cycling (e.g., 92.5 for 92.5%)."
    )
    activation_energy_ev: Optional[float] = Field(
        None,
        description="Activation energy in eV, typically derived from Arrhenius plot slope."
    )

# Verify the new schema
print("Extended schema fields:")
for name, fi in ExtendedMeasuredPoint.model_fields.items():
    if name in ("electrode_material", "cycle_count", "capacity_retention_pct", "activation_energy_ev"):
        print(f"  + {name:<30} (NEW)")
    # Only print first few base fields to keep output concise
    elif name in ("raw_composition", "normalized_conductivity", "source"):
        print(f"    {name:<30} (base)")

print(f"\nTotal fields: {len(ExtendedMeasuredPoint.model_fields)} "
      f"(base: {len(MeasuredPoint.model_fields)}, added: "
      f"{len(ExtendedMeasuredPoint.model_fields) - len(MeasuredPoint.model_fields)})")

### 3b. Replacing the schema for a different domain

For a completely different extraction task (e.g., mechanical properties, catalytic performance), define a new Pydantic model from scratch. The field names and descriptions are what guide the LLM — the more precise your descriptions, the better the extraction.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List

class MechanicalPropertyPoint(BaseModel):
    """Schema for extracting mechanical test data from materials science papers."""
    
    # --- Material identity ---
    raw_composition: str = Field(
        ..., description="Material name as it appears in the source text."
    )
    canonical_formula: Optional[str] = Field(
        None, description="Standardized chemical formula or material designation."
    )
    material_class: Optional[str] = Field(
        None, description="Material category: 'Metal', 'Ceramic', 'Polymer', 'Composite'."
    )
    
    # --- Mechanical properties ---
    tensile_strength_mpa: Optional[float] = Field(
        None, description="Ultimate tensile strength in MPa."
    )
    yield_strength_mpa: Optional[float] = Field(
        None, description="Yield strength (0.2% offset) in MPa."
    )
    elastic_modulus_gpa: Optional[float] = Field(
        None, description="Young's modulus / elastic modulus in GPa."
    )
    elongation_at_break_pct: Optional[float] = Field(
        None, description="Elongation at break as percentage (e.g., 15.2 for 15.2%)."
    )
    hardness: Optional[str] = Field(
        None, description="Hardness value with scale (e.g., '65 HRC', '350 HV')."
    )
    
    # --- Test conditions ---
    test_temperature_c: Optional[float] = Field(
        None, description="Test temperature in Celsius."
    )
    strain_rate: Optional[str] = Field(
        None, description="Strain rate as reported (e.g., '1e-3 s^-1', '0.5 mm/min')."
    )
    test_standard: Optional[str] = Field(
        None, description="Testing standard followed (e.g., 'ASTM E8', 'ISO 6892-1')."
    )
    
    # --- Provenance ---
    source: str = Field(
        ..., description="Source type: 'text', 'table', 'figure'."
    )
    confidence: str = Field(
        ..., description="Extraction confidence: 'high', 'medium', 'low'."
    )

# Preview the JSON Schema that would be sent to the LLM
import json
schema = MechanicalPropertyPoint.model_json_schema()
print(json.dumps(schema, indent=2)[:1500])
print(f"\n... ({len(schema['properties'])} fields total)")

### 3c. How schema changes propagate to prompts

When you modify the schema, the LLM receives the updated JSON Schema as part of its structured output specification. Here's what happens under the hood:

```
Your Pydantic Model
    │
    ▼
model.model_json_schema()  →  JSON Schema (field names, types, descriptions)
    │
    ▼
Gemini API call with response_schema=...  →  LLM extracts into your fields
    │
    ▼
Pydantic validation  →  Type-checked, validated output
```

**Key insight:** The field *descriptions* are your prompts. A field named `tensile_strength_mpa` with description `"Ultimate tensile strength in MPa"` tells the LLM exactly what to look for. The more specific your descriptions (with examples, units, edge cases), the better the extraction quality.

> To use a custom schema with the pipeline, you would swap `MeasuredPoint` in `extraction_logic.py` with your new model. The extraction prompts in `process_text`, `process_table_node`, and `process_image` reference the schema fields — so updating those prompts to match your new domain is the main integration step.

---
## 4. Inspect the Extraction Prompts

The pipeline uses specialized prompts for each modality (text, tables, figures). You can read and modify these prompts to tune extraction behavior.

In [ ]:
from parse import show_prompts

prompts = show_prompts()

# Available prompt categories
print("Prompt categories:")
for key, info in prompts.items():
    print(f"  {key:<30} ({info['source_lines']} lines — {info['function']})")

print("\n--- To read a specific prompt, e.g.: ---")
print("print(prompts['text_extraction']['source'][:2000])")

In [ ]:
# Read the text extraction prompt (the core workhorse)
# This shows how the LLM is instructed to find conductivity data in text sections
print(prompts["text_extraction"]["source"][:3000])
print("\n... (truncated — full prompt is", prompts["text_extraction"]["source_lines"], "lines)")

---
## 5. Run Extraction with Verification

The **agentic verifier** is a separate LLM agent that:
1. Reads the structured execution log from the pipeline
2. Identifies failures (timeouts, schema errors, missing data)
3. Re-runs only the failed components
4. Does a full-paper PDF sweep to catch missed or hallucinated measurements

In [ ]:
# Run extraction + verification in one call
paper = "../output/downselectedpapers_jiyoung/Composite Polymer Electrolytes with Li7La3Zr2O12 Garnet-Type Nanowires as Ceramic Fillers/Composite Polymer Electrolytes with Li7La3Zr2O12 Garnet-Type Nanowires as Ceramic Fillers.md"
pdf   = "../output/downselectedpapers_jiyoung/Composite Polymer Electrolytes with Li7La3Zr2O12 Garnet-Type Nanowires as Ceramic Fillers/Composite Polymer Electrolytes with Li7La3Zr2O12 Garnet-Type Nanowires as Ceramic Fillers.pdf"

result = extract(paper, pdf=pdf, verify=True)

In [ ]:
# View the verifier's report
if result.verification_report:
    print(result.verification_report)
else:
    print("No verification report (run with verify=True to enable)")

---
## 6. Explore Results

Dig into the extracted measurements — filter by source modality, material class, temperature range, etc.

In [ ]:
import pandas as pd

df = result.dataframe

# Summary by source modality
print("Measurements by source:")
print(df["source"].value_counts().to_string())
print()

# Summary by material class
print("Measurements by material class:")
print(df["material_class"].value_counts().to_string())
print()

# Summary by confidence
print("Measurements by confidence:")
print(df["confidence"].value_counts().to_string())

In [ ]:
# Filter: room-temperature measurements only
rt_mask = df["normalized_temperature_c"].between(15, 35)
rt_df = df[rt_mask].sort_values("normalized_conductivity", ascending=False)

print(f"Room-temperature measurements: {len(rt_df)} / {len(df)}")
rt_df[["canonical_formula", "material_class", "normalized_conductivity", 
       "normalized_temperature_c", "source", "confidence"]]

In [ ]:
# Provenance: trace where each measurement came from
for i, row in df.head(3).iterrows():
    print(f"Measurement {i}:")
    print(f"  Material  : {row['canonical_formula']}")
    print(f"  sigma     : {row['normalized_conductivity']:.2e} S/cm at {row['normalized_temperature_c']}C")
    print(f"  Source    : {row['source']} (section: {row.get('source_section', 'N/A')})")
    if row.get("source_figure_id"):
        print(f"  Figure    : {row['source_figure_id']}")
    if row.get("source_paragraph_id"):
        print(f"  Paragraph : {row['source_paragraph_id']}")
    print()

---
## 7. Visualize Extraction Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Panel 1: Measurements by source ---
source_counts = df["source"].value_counts()
colors = {"text": "#4C78A8", "figure": "#E45756", "table": "#54A24B",
          "cited_text": "#88BEDC", "cited_table": "#A0CBE8", "cited_figure": "#F2A2A0"}
bar_colors = [colors.get(s, "#999") for s in source_counts.index]
axes[0].barh(source_counts.index, source_counts.values, color=bar_colors)
axes[0].set_xlabel("Count")
axes[0].set_title("Measurements by Source Modality")
axes[0].invert_yaxis()

# --- Panel 2: Conductivity distribution ---
valid = df["normalized_conductivity"].dropna()
if len(valid) > 0:
    log_sigma = np.log10(valid[valid > 0])
    axes[1].hist(log_sigma, bins=15, color="#4C78A8", edgecolor="white", alpha=0.8)
    axes[1].set_xlabel("log10(sigma / S cm$^{-1}$)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Conductivity Distribution")

# --- Panel 3: Confidence breakdown ---
conf_counts = df["confidence"].value_counts()
conf_colors = {"high": "#54A24B", "medium": "#F5A623", "low": "#E45756"}
axes[2].pie(conf_counts.values, 
            labels=conf_counts.index, 
            colors=[conf_colors.get(c, "#999") for c in conf_counts.index],
            autopct="%1.0f%%", startangle=90)
axes[2].set_title("Extraction Confidence")

plt.suptitle(f"Extraction Summary: {result.doc_name[:60]}", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Batch Extraction — Multiple Papers

In [ ]:
from pathlib import Path
from parse import extract

# Find all markdown papers in a directory
papers_dir = Path("../output/downselectedpapers_jiyoung/")
paper_mds = sorted(papers_dir.glob("*/*.md"))

print(f"Found {len(paper_mds)} papers:")
for p in paper_mds:
    print(f"  {p.parent.name[:80]}")

In [ ]:
# Extract from multiple papers (saves results automatically)
# Uncomment to run — this will make API calls for each paper
#
# results = extract(
#     [str(p) for p in paper_mds],
#     result_save_dir="./batch_results/"
# )
#
# # Aggregate summary
# for r in results:
#     print(f"{r.doc_name[:50]:50s}  {r.n_measurements:3d} measurements  ${r.cost['total_usd']:.4f}")
#
# total_measurements = sum(r.n_measurements for r in results)
# total_cost = sum(r.cost['total_usd'] for r in results)
# print(f"\nTotal: {total_measurements} measurements from {len(results)} papers — ${total_cost:.4f}")

---
## 9. Save & Export

In [ ]:
# Save as JSON (full results with provenance, cost, and config)
saved_path = result.save("./my_extraction_results.json")
print(f"Saved to: {saved_path}")

# Save as CSV (measurements table only, for spreadsheet analysis)
df = result.dataframe
df.to_csv("./my_extraction_results.csv", index=False)
print(f"CSV saved: {len(df)} rows")

---
## 10. Load & Inspect Pre-Cached Results

If you don't have an API key or want to skip the extraction step, you can load pre-cached results from papers we've already processed.

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Load a pre-cached result
cached_path = Path("../output/downselectedpapers_jiyoung/"
                   "Composite Polymer Electrolytes with Li7La3Zr2O12 "
                   "Garnet-Type Nanowires as Ceramic Fillers/"
                   "robust_results_v8.json")

with open(cached_path) as f:
    cached = json.load(f)

print(f"Paper: {cached['doc_name']}")
print(f"Measurements: {cached['material_count']}")
print(f"Execution time: {cached['execution_time_seconds']:.1f}s")
print(f"API cost: ${cached['cost_summary']['total_cost_usd']:.4f}")
print(f"Models used: {cached['config']}")

# Convert to DataFrame for analysis
df_cached = pd.DataFrame(cached["measurements"])
df_cached[["canonical_formula", "material_class", "normalized_conductivity", 
           "normalized_temperature_c", "source", "confidence"]]

In [ ]:
# Load the verification report
verif_path = cached_path.parent / "verification_report.json"
if verif_path.exists():
    with open(verif_path) as f:
        verif = json.load(f)
    
    print(f"Missing candidates found by verifier: {len(verif.get('missing_candidates', []))}")
    print(f"Potential hallucinations flagged:      {len(verif.get('hallucinations', []))}")
    print()
    
    # Show what the verifier caught
    for mc in verif.get("missing_candidates", [])[:3]:
        print(f"  MISSING: {mc['composition']}")
        print(f"           {mc['conductivity']} at {mc['temperature']} ({mc['location']})")
        print()

---
## 11. Pipeline Architecture Reference

```
                                    PageIndex Pipeline
                                    ==================

    ┌─────────────┐
    │  Paper PDF  │
    └──────┬──────┘
           │  Document Ingestion (PDF → Markdown + assets)
           ▼
    ┌─────────────┐     ┌──────────────────────────────────────────────────┐
    │  Paper .md  │────▶│  Phase 1: Multi-Modal Extraction                │
    │  + images   │     │                                                  │
    │  + tables   │     │  ┌────────────┐ ┌──────────────┐ ┌────────────┐ │
    └─────────────┘     │  │ Text       │ │ Tables       │ │ Figures    │ │
                        │  │ Extractor  │ │ Extractor    │ │ SciFigure  │ │
                        │  │ (sections) │ │ (markdown)   │ │ Parser     │ │
                        │  └─────┬──────┘ └──────┬───────┘ └─────┬──────┘ │
                        │        │               │               │        │
                        │        └───────┬───────┴───────┬───────┘        │
                        │                ▼               ▼                │
                        │        ┌──────────────┐ ┌────────────┐          │
                        │        │ Normalize    │ │ QA + Dedup │          │
                        │        │ (units, T)   │ │ (validate) │          │
                        │        └──────┬───────┘ └─────┬──────┘          │
                        └───────────────┼───────────────┼─────────────────┘
                                        ▼               ▼
                                 ┌─────────────────────────┐
                                 │  Extraction Results      │
                                 │  + extraction_log.jsonl  │
                                 └────────────┬────────────┘
                                              │
                        ┌─────────────────────▼──────────────────────┐
                        │  Phase 2: Agentic Verification             │
                        │                                            │
                        │  1. Read execution log                     │
                        │  2. Identify failures (timeout, schema,    │
                        │     missing data, normalization)           │
                        │  3. Re-run failed components only          │
                        │  4. Full-paper PDF sweep                   │
                        │  5. Flag hallucinations + missing data     │
                        └────────────────────┬───────────────────────┘
                                             ▼
                                 ┌───────────────────────┐
                                 │  Validated Results     │
                                 │  + verification_report │
                                 └───────────────────────┘
```

**Key design decisions:**
- **Parallel multi-modal extraction**: Text, tables, and figures are processed independently, then merged. This is more robust than sequential processing where one failure blocks everything.
- **Agency in verification, not extraction**: The extraction itself is a deterministic pipeline (reproducible, efficient). The *agency* is in the error-recovery loop where an LLM reasons about failures and decides what to retry and how.
- **Schema-driven extraction**: The Pydantic model's field descriptions serve as natural-language specifications that guide the LLM. Changing the schema changes what gets extracted.